# Diabetes Healthcare Analysis & Readmission Prediction

## Project Objective

The objective of this project is to analyse hospital encounters from the Diabetes 130-US Hospitals dataset (1999–2008) to identify factors associated with hospital readmission within 30 days.

The project combines data cleaning, exploratory data analysis, statistical hypothesis testing, feature engineering and predictive modelling. Logistic Regression and Random Forest classifiers are developed and compared to assess their ability to identify encounters followed by a readmission within 30 days.

The analysis examines demographic characteristics, clinical information, medication-related variables and previous healthcare utilisation. Model performance is evaluated using multiple metrics, including ROC-AUC, precision, recall and F1 score, with particular consideration given to the imbalance between readmitted and non-readmitted encounters.

The goal is to demonstrate a complete healthcare analytics workflow and critically assess the usefulness and limitations of predictive modelling for readmission analysis.

## Dataset Overview

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/diabetic_data.csv")

df.shape

(101766, 50)

In [2]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

## Data Inspection

In [4]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate rows:", df.duplicated().sum())

Rows: 101766
Columns: 50
Duplicate rows: 0


In [5]:
df.isna().sum().sort_values(ascending=False).head(15)

max_glu_serum    96420
A1Cresult        84748
encounter_id         0
nateglinide          0
glimepiride          0
acetohexamide        0
glipizide            0
glyburide            0
tolbutamide          0
pioglitazone         0
rosiglitazone        0
acarbose             0
miglitol             0
troglitazone         0
tolazamide           0
dtype: int64

In [6]:
df["readmitted"].value_counts(dropna=False)

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [7]:
missing_marker_counts = (df == "?").sum().sort_values(ascending=False)

missing_marker_counts.head(20)

weight               98569
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
encounter_id             0
tolazamide               0
glyburide                0
tolbutamide              0
pioglitazone             0
rosiglitazone            0
acarbose                 0
miglitol                 0
troglitazone             0
citoglipton              0
examide                  0
acetohexamide            0
insulin                  0
dtype: int64

In [8]:
missing_marker_percent = (
    (df == "?").mean() * 100
).sort_values(ascending=False)

missing_marker_percent.head(20)

weight               96.858479
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
encounter_id          0.000000
tolazamide            0.000000
glyburide             0.000000
tolbutamide           0.000000
pioglitazone          0.000000
rosiglitazone         0.000000
acarbose              0.000000
miglitol              0.000000
troglitazone          0.000000
citoglipton           0.000000
examide               0.000000
acetohexamide         0.000000
insulin               0.000000
dtype: float64

In [9]:
for column in ["weight", "payer_code", "medical_specialty", "race", "gender"]:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False).head(10))


weight
weight
?            98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64

payer_code
payer_code
?     40256
MC    32439
HM     6274
SP     5007
BC     4655
MD     3532
CP     2533
UN     2448
CM     1937
OG     1033
Name: count, dtype: int64

medical_specialty
medical_specialty
?                             49949
InternalMedicine              14635
Emergency/Trauma               7565
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Radiologist                    1140
Name: count, dtype: int64

race
race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dty

In [10]:
id_columns = ["encounter_id", "patient_nbr"]

for column in id_columns:
    print(f"\n{column}")
    print("Unique values:", df[column].nunique())
    print("Duplicate values:", df[column].duplicated().sum())


encounter_id
Unique values: 101766
Duplicate values: 0

patient_nbr
Unique values: 71518
Duplicate values: 30248


In [11]:
categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False).head(10))


--- race ---
race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

--- gender ---
gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

--- age ---
age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64

--- weight ---
weight
?            98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64

--- payer_code ---
payer_code
?     40256
MC    32439
HM     6274
SP     5007
BC     4655
MD     3532
CP     2533
UN     2448
CM     1937
OG     1033
Name: count, dtype: int64

--- medical_specialty ---
medical_special

## Data Cleaning

In [12]:
import numpy as np

df_clean = df.copy()

# Preserve patient identifier for later patient-level splitting.
# It will not be used as a model feature.

# Convert dataset-specific missing markers to NaN
missing_marker_columns = [
    "race",
    "weight",
    "payer_code",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3"
]

df_clean[missing_marker_columns] = (
    df_clean[missing_marker_columns]
    .replace("?", np.nan)
)

df_clean.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [13]:
# Remove variables with extremely high missingness
df_clean = df_clean.drop(columns=["weight"])

# Remove constant variables
constant_columns = [
    column
    for column in df_clean.columns
    if df_clean[column].nunique(dropna=False) <= 1
]

constant_columns

['examide', 'citoglipton']

In [14]:
df_clean = df_clean.drop(columns=constant_columns)

print("Removed constant columns:", constant_columns)
print("New shape:", df_clean.shape)

Removed constant columns: ['examide', 'citoglipton']
New shape: (101766, 47)


In [15]:
categorical_missing_columns = [
    "race",
    "payer_code",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3"
]

df_clean[categorical_missing_columns] = (
    df_clean[categorical_missing_columns]
    .fillna("Unknown")
)

In [16]:
df_clean[categorical_missing_columns].isna().sum()

race                 0
payer_code           0
medical_specialty    0
diag_1               0
diag_2               0
diag_3               0
dtype: int64

In [17]:
age_mapping = {
    "[0-10)": 5,
    "[10-20)": 15,
    "[20-30)": 25,
    "[30-40)": 35,
    "[40-50)": 45,
    "[50-60)": 55,
    "[60-70)": 65,
    "[70-80)": 75,
    "[80-90)": 85,
    "[90-100)": 95
}

df_clean["age_numeric"] = df_clean["age"].map(age_mapping)

In [18]:
df_clean[["age", "age_numeric"]].drop_duplicates().sort_values("age_numeric")

,age,age_numeric
0,[0-10),5
1,[10-20),15
2,[20-30),25
3,[30-40),35
4,[40-50),45
5,[50-60),55
6,[60-70),65
7,[70-80),75
8,[80-90),85
9,[90-100),95


In [19]:
df_clean["readmission_30d"] = (
    df_clean["readmitted"] == "<30"
).astype(int)

df_clean["readmission_30d"].value_counts()

readmission_30d
0    90409
1    11357
Name: count, dtype: int64

In [20]:
pd.crosstab(
    df_clean["readmitted"],
    df_clean["readmission_30d"]
)

readmission_30d,0,1
readmitted,,
<30,0,11357
>30,35545,0
NO,54864,0


In [21]:
for column in ["diag_1", "diag_2", "diag_3"]:
    print(
        column,
        "unique codes:",
        df_clean[column].nunique()
    )

diag_1 unique codes: 717
diag_2 unique codes: 749
diag_3 unique codes: 790


In [22]:
df_clean[["diag_1", "diag_2", "diag_3"]].head(10)

,diag_1,diag_2,diag_3
0,250.83,Unknown,Unknown
1,276,250.01,255
2,648,250,V27
3,8,250.43,403
4,197,157,250
5,414,411,250
6,414,411,V45
7,428,492,250
8,398,427,38
9,434,198,486


In [23]:
def get_icd9_category(code):
    if pd.isna(code) or code == "Unknown":
        return "Unknown"
    
    code = str(code)
    
    if code.startswith("V"):
        return "V-code"
    elif code.startswith("E"):
        return "E-code"
    
    try:
        numeric_code = float(code)
        
        if 390 <= numeric_code < 460:
            return "Circulatory"
        elif 460 <= numeric_code < 520:
            return "Respiratory"
        elif 520 <= numeric_code < 580:
            return "Digestive"
        elif 580 <= numeric_code < 630:
            return "Genitourinary"
        elif 630 <= numeric_code < 680:
            return "Pregnancy"
        elif 680 <= numeric_code < 710:
            return "Skin"
        elif 710 <= numeric_code < 740:
            return "Musculoskeletal"
        elif 740 <= numeric_code < 760:
            return "Congenital"
        elif 760 <= numeric_code < 780:
            return "Perinatal"
        elif 780 <= numeric_code < 800:
            return "Symptoms"
        elif 800 <= numeric_code < 1000:
            return "Injury"
        elif 250 <= numeric_code < 251:
            return "Diabetes"
        else:
            return "Other"
            
    except ValueError:
        return "Other"

In [24]:
for column in ["diag_1", "diag_2", "diag_3"]:
    df_clean[f"{column}_category"] = df_clean[column].apply(get_icd9_category)

In [25]:
for column in ["diag_1_category", "diag_2_category", "diag_3_category"]:
    print(f"\n--- {column} ---")
    print(df_clean[column].value_counts())


--- diag_1_category ---
diag_1_category
Circulatory        30336
Other              13479
Respiratory        10407
Digestive           9208
Diabetes            8757
Symptoms            7636
Injury              6974
Genitourinary       5078
Musculoskeletal     4957
Skin                2530
V-code              1644
Pregnancy            687
Congenital            51
Unknown               21
E-code                 1
Name: count, dtype: int64

--- diag_2_category ---
diag_2_category
Circulatory        31365
Other              19570
Diabetes           12794
Respiratory        10251
Genitourinary       7987
Symptoms            4632
Digestive           3962
Skin                3596
Injury              2428
V-code              1805
Musculoskeletal     1764
E-code               731
Pregnancy            415
Unknown              358
Congenital           108
Name: count, dtype: int64

--- diag_3_category ---
diag_3_category
Circulatory        29918
Other              20260
Diabetes           17157


In [26]:
df_clean.shape

(101766, 52)

In [27]:
df_clean.isna().sum().sort_values(ascending=False).head(15)

max_glu_serum          96420
A1Cresult              84748
encounter_id               0
insulin                    0
glipizide                  0
glyburide                  0
tolbutamide                0
pioglitazone               0
rosiglitazone              0
acarbose                   0
miglitol                   0
troglitazone               0
tolazamide                 0
glyburide-metformin        0
glimepiride                0
dtype: int64

In [28]:
df_clean.to_csv(
    "../data/cleaned/diabetes_cleaned_intermediate.csv",
    index=False
)

print("Saved:", df_clean.shape)

Saved: (101766, 52)


## Exploratory Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_analysis = pd.read_csv(
    "../data/cleaned/diabetes_cleaned_intermediate.csv"
)

print("Shape:", df_analysis.shape)
df_analysis.head()

In [ ]:
readmission_counts = df_analysis["readmission_30d"].value_counts()
readmission_rates = df_analysis["readmission_30d"].value_counts(normalize=True) * 100

print("Readmission counts:")
print(readmission_counts)

print("\nReadmission rates:")
print(readmission_rates.round(2))

In [ ]:
readmission_summary = pd.DataFrame({
    "count": readmission_counts,
    "percentage": readmission_rates.round(2)
})

readmission_summary

In [ ]:
age_summary = (
    df_analysis
    .groupby("readmission_30d")["age_numeric"]
    .agg(["count", "mean", "median", "std"])
)

age_summary

In [ ]:
plt.figure(figsize=(8, 5))

df_analysis.boxplot(
    column="age_numeric",
    by="readmission_30d"
)

plt.title("Age Distribution by 30-Day Readmission")
plt.suptitle("")
plt.xlabel("30-Day Readmission (0 = No, 1 = Yes)")
plt.ylabel("Age")
plt.show()

In [ ]:
utilisation_summary = (
    df_analysis
    .groupby("readmission_30d")[
        ["number_inpatient", "number_emergency", "number_outpatient"]
    ]
    .agg(["mean", "median", "std"])
)

utilisation_summary

In [ ]:
utilisation_means = (
    df_analysis
    .groupby("readmission_30d")[
        ["number_inpatient", "number_emergency", "number_outpatient"]
    ]
    .mean()
    .round(2)
)

utilisation_means

In [ ]:
los_summary = (
    df_analysis
    .groupby("readmission_30d")["time_in_hospital"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

los_summary

In [ ]:
plt.figure(figsize=(8, 5))

df_analysis.boxplot(
    column="time_in_hospital",
    by="readmission_30d"
)

plt.title("Length of Stay by 30-Day Readmission")
plt.suptitle("")
plt.xlabel("30-Day Readmission (0 = No, 1 = Yes)")
plt.ylabel("Days in Hospital")
plt.show()

### Statistical Comparison of Continuous Variables

In [ ]:
from scipy.stats import ttest_ind

continuous_variables = [
    "age_numeric",
    "time_in_hospital",
    "number_inpatient",
    "number_emergency",
    "number_outpatient"
]

statistical_results = []

for variable in continuous_variables:
    no_readmission = df_analysis.loc[
        df_analysis["readmission_30d"] == 0,
        variable
    ].dropna()
    
    readmission = df_analysis.loc[
        df_analysis["readmission_30d"] == 1,
        variable
    ].dropna()
    
    t_stat, p_value = ttest_ind(
        no_readmission,
        readmission,
        equal_var=False
    )
    
    statistical_results.append({
        "variable": variable,
        "mean_no_readmission": no_readmission.mean(),
        "mean_readmission": readmission.mean(),
        "mean_difference": readmission.mean() - no_readmission.mean(),
        "t_statistic": t_stat,
        "p_value": p_value
    })

statistical_results = pd.DataFrame(statistical_results)

statistical_results

In [ ]:
def cohens_d(group1, group2):
    n1 = len(group1)
    n2 = len(group2)
    
    var1 = group1.var(ddof=1)
    var2 = group2.var(ddof=1)
    
    pooled_sd = np.sqrt(
        ((n1 - 1) * var1 + (n2 - 1) * var2)
        / (n1 + n2 - 2)
    )
    
    return (group2.mean() - group1.mean()) / pooled_sd


effect_sizes = []

for variable in continuous_variables:
    no_readmission = df_analysis.loc[
        df_analysis["readmission_30d"] == 0,
        variable
    ].dropna()
    
    readmission = df_analysis.loc[
        df_analysis["readmission_30d"] == 1,
        variable
    ].dropna()
    
    effect_sizes.append({
        "variable": variable,
        "cohens_d": cohens_d(no_readmission, readmission)
    })

effect_sizes = pd.DataFrame(effect_sizes)

effect_sizes

### Categorical Variables and 30-Day Readmission

In [ ]:
categorical_variables = [
    "gender",
    "race",
    "age",
    "diabetesMed",
    "change"
]

categorical_summaries = {}

for variable in categorical_variables:
    summary = (
        pd.crosstab(
            df_analysis[variable],
            df_analysis["readmission_30d"],
            normalize="index"
        ) * 100
    )
    
    summary.columns = [
        "No readmission (%)",
        "30-day readmission (%)"
    ]
    
    categorical_summaries[variable] = summary.round(2)

categorical_summaries["gender"]

In [ ]:
categorical_summaries["race"]

In [ ]:
categorical_summaries["age"]

In [ ]:
categorical_summaries["diabetesMed"]

In [ ]:
categorical_summaries["change"]

In [ ]:
from scipy.stats import chi2_contingency

chi_square_results = []

for variable in categorical_variables:
    contingency_table = pd.crosstab(
        df_analysis[variable],
        df_analysis["readmission_30d"]
    )
    
    chi2, p_value, dof, expected = chi2_contingency(
        contingency_table
    )
    
    chi_square_results.append({
        "variable": variable,
        "chi2": chi2,
        "degrees_of_freedom": dof,
        "p_value": p_value
    })

chi_square_results = pd.DataFrame(chi_square_results)

chi_square_results

In [ ]:
def cramers_v(contingency_table):
    chi2, _, _, _ = chi2_contingency(contingency_table)
    
    n = contingency_table.sum().sum()
    r, k = contingency_table.shape
    
    phi2 = chi2 / n
    
    phi2_corrected = max(
        0,
        phi2 - ((k - 1) * (r - 1)) / (n - 1)
    )
    
    r_corrected = r - ((r - 1) ** 2) / (n - 1)
    k_corrected = k - ((k - 1) ** 2) / (n - 1)
    
    denominator = min(
        k_corrected - 1,
        r_corrected - 1
    )
    
    return np.sqrt(phi2_corrected / denominator)


cramers_results = []

for variable in categorical_variables:
    contingency_table = pd.crosstab(
        df_analysis[variable],
        df_analysis["readmission_30d"]
    )
    
    cramers_results.append({
        "variable": variable,
        "cramers_v": cramers_v(contingency_table)
    })

cramers_results = pd.DataFrame(cramers_results)

cramers_results

### Clinical and Encounter Characteristics

In [ ]:
clinical_variables = [
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_diagnoses"
]

clinical_summary = (
    df_analysis
    .groupby("readmission_30d")[clinical_variables]
    .agg(["mean", "median", "std"])
)

clinical_summary.round(2)

In [ ]:
clinical_results = []

for variable in clinical_variables:
    no_readmission = df_analysis.loc[
        df_analysis["readmission_30d"] == 0,
        variable
    ].dropna()
    
    readmission = df_analysis.loc[
        df_analysis["readmission_30d"] == 1,
        variable
    ].dropna()
    
    t_stat, p_value = ttest_ind(
        no_readmission,
        readmission,
        equal_var=False
    )
    
    d = cohens_d(no_readmission, readmission)
    
    clinical_results.append({
        "variable": variable,
        "mean_no_readmission": no_readmission.mean(),
        "mean_readmission": readmission.mean(),
        "mean_difference": readmission.mean() - no_readmission.mean(),
        "t_statistic": t_stat,
        "p_value": p_value,
        "cohens_d": d
    })

clinical_results = pd.DataFrame(clinical_results)

clinical_results.round(4)

In [ ]:
for variable in ["max_glu_serum", "A1Cresult"]:
    print(f"\n{variable}")
    
    summary = (
        pd.crosstab(
            df_analysis[variable],
            df_analysis["readmission_30d"],
            normalize="index"
        ) * 100
    )
    
    summary.columns = [
        "No readmission (%)",
        "30-day readmission (%)"
    ]
    
    print(summary.round(2))

In [ ]:
measurement_variables = ["max_glu_serum", "A1Cresult"]

for variable in measurement_variables:
    measurement_status = df_analysis[variable].fillna("Not measured")
    
    summary = (
        pd.crosstab(
            measurement_status,
            df_analysis["readmission_30d"],
            normalize="index"
        ) * 100
    )
    
    summary.columns = [
        "No readmission (%)",
        "30-day readmission (%)"
    ]
    
    print(f"\n{variable}")
    display(summary.round(2))

In [ ]:
primary_diagnosis_summary = (
    pd.crosstab(
        df_analysis["diag_1_category"],
        df_analysis["readmission_30d"],
        normalize="index"
    ) * 100
)

primary_diagnosis_summary.columns = [
    "No readmission (%)",
    "30-day readmission (%)"
]

primary_diagnosis_summary = (
    primary_diagnosis_summary
    .sort_values("30-day readmission (%)", ascending=False)
)

primary_diagnosis_summary.round(2)

In [ ]:
primary_diagnosis_counts = (
    df_analysis["diag_1_category"]
    .value_counts()
    .rename("encounters")
)

primary_diagnosis_readmissions = (
    df_analysis
    .groupby("diag_1_category")["readmission_30d"]
    .agg(["sum", "mean"])
    .rename(columns={
        "sum": "readmissions",
        "mean": "readmission_rate"
    })
)

primary_diagnosis_analysis = (
    primary_diagnosis_counts
    .to_frame()
    .join(primary_diagnosis_readmissions)
)

primary_diagnosis_analysis["readmission_rate"] *= 100

primary_diagnosis_analysis = (
    primary_diagnosis_analysis
    .sort_values("readmission_rate", ascending=False)
)

primary_diagnosis_analysis.round(2)

In [ ]:
diagnosis_contingency = pd.crosstab(
    df_analysis["diag_1_category"],
    df_analysis["readmission_30d"]
)

chi2, p_value, dof, expected = chi2_contingency(
    diagnosis_contingency
)

diagnosis_cramers_v = cramers_v(diagnosis_contingency)

print("Chi-square statistic:", round(chi2, 3))
print("Degrees of freedom:", dof)
print("p-value:", p_value)
print("Cramér's V:", round(diagnosis_cramers_v, 4))

In [ ]:
diagnosis_plot = (
    primary_diagnosis_analysis[
        primary_diagnosis_analysis["encounters"] >= 1000
    ]
    .sort_values("readmission_rate")
)

plt.figure(figsize=(10, 6))

plt.barh(
    diagnosis_plot.index,
    diagnosis_plot["readmission_rate"]
)

plt.xlabel("30-Day Readmission Rate (%)")
plt.ylabel("Primary Diagnosis Category")
plt.title("30-Day Readmission Rate by Primary Diagnosis Category")

plt.axvline(
    df_analysis["readmission_30d"].mean() * 100,
    linestyle="--",
    label="Overall readmission rate"
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
statistical_results.to_csv(
    "../data/cleaned/continuous_readmission_statistics.csv",
    index=False
)

clinical_results.to_csv(
    "../data/cleaned/clinical_readmission_statistics.csv",
    index=False
)

chi_square_results.to_csv(
    "../data/cleaned/categorical_chi_square_results.csv",
    index=False
)

cramers_results.to_csv(
    "../data/cleaned/categorical_effect_sizes.csv",
    index=False
)

primary_diagnosis_analysis.to_csv(
    "../data/cleaned/primary_diagnosis_readmission_rates.csv"
)

print("Statistical result files saved.")

### Exploratory Analysis Findings

The dataset contained 101,766 hospital encounters, of which 11,357 (11.16%) were followed by a readmission within 30 days.

Among continuous variables examined, prior inpatient utilisation showed the strongest association with 30-day readmission. Patients with a 30-day readmission had an average of 1.22 prior inpatient visits compared with 0.56 among other encounters, corresponding to a moderate standardised difference (Cohen's d = 0.532).

Length of stay, prior emergency utilisation, medication count and number of diagnoses also differed statistically between groups, but their effect sizes were small. Age, laboratory procedures and outpatient utilisation had negligible effect sizes despite statistically significant differences.

Categorical variables generally showed weak associations with readmission. Gender had essentially no association, while race, age group, diabetes medication status and medication change all had negligible Cramér's V values.

Primary diagnosis category was statistically associated with 30-day readmission (χ² = 165.853, p < 0.001), but the effect size was small (Cramér's V = 0.039). Descriptively, diabetes, injury and V-code primary diagnoses had higher readmission rates than the overall 11.16% baseline, although small diagnosis groups should be interpreted cautiously.

Glucose and HbA1c measurement categories showed relatively modest differences in readmission rates. The high proportion of encounters without recorded glucose or HbA1c results was retained as a distinct "Not measured" category for analysis rather than assuming that missingness represented random missing data.

Overall, the exploratory analysis suggests that prior healthcare utilisation, particularly prior inpatient utilisation, may contain more useful information about 30-day readmission risk than most individual demographic or categorical characteristics examined.

## Feature Engineering

### Prediction Point and Leakage Control

The predictive task is to estimate whether a hospital encounter will be followed by a readmission within 30 days.

The prediction point is defined as during the hospital encounter, before discharge. Predictor variables should therefore represent information that could reasonably be available during the encounter without using information about the eventual readmission outcome.

The following variables are excluded from modelling:

- `readmission_30d`: the target variable.
- `readmitted`: the original readmission outcome variable.
- `encounter_id`: an encounter identifier rather than a clinical predictor.
- `patient_nbr`: a patient identifier. It is retained temporarily for patient-level train/test splitting but is not used as a model feature.
- `discharge_disposition_id`: information associated with discharge and therefore potentially unavailable at the defined prediction point.

The modelling dataset will retain clinically and operationally relevant variables that could reasonably be available before discharge.

In [ ]:
# Create a copy for feature engineering
df_model = df_analysis.copy()

# Variables excluded from model features
excluded_features = [
    "readmission_30d",
    "readmitted",
    "encounter_id",
    "discharge_disposition_id"
]

# Keep patient_nbr temporarily for patient-level splitting
model_columns = [
    col for col in df_model.columns
    if col not in excluded_features
]

df_model = df_model[model_columns].copy()

print("Modelling dataset shape:", df_model.shape)
print("\nColumns retained:")
print(df_model.columns.tolist())

In [ ]:
# Separate numeric and categorical variables
numeric_features = df_model.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = df_model.select_dtypes(
    include=["object"]
).columns.tolist()

print("Number of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

### Feature Selection

The original ICD-9 diagnosis variables contain hundreds of distinct codes and would create a high-dimensional categorical feature space. The engineered diagnosis-category variables provide a more interpretable representation of broad clinical conditions and are therefore retained instead.

The patient identifier `patient_nbr` is retained temporarily for patient-level splitting but will be removed before modelling.

The original diagnosis variables `diag_1`, `diag_2` and `diag_3` are excluded from the model features.

In [ ]:
# Remove patient identifier and high-cardinality raw diagnosis codes
features_to_exclude = [
    "patient_nbr",
    "diag_1",
    "diag_2",
    "diag_3"
]

X = df_model.drop(columns=features_to_exclude)

y = df_analysis["readmission_30d"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget proportion:")
print(y.value_counts(normalize=True).round(4))

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nNumeric:")
print(numeric_features)

print("\nCategorical:")
print(categorical_features)

### Patient-Level Train/Test Split

Because patients may have multiple hospital encounters, an encounter-level random split could place encounters from the same patient in both the training and test datasets.

To reduce this potential leakage, the train/test split is performed at the patient level. Each patient is assigned entirely to either the training or test set.

The `patient_nbr` identifier is used only to construct this split and is not included as a model feature.

In [ ]:
from sklearn.model_selection import train_test_split

# Patient identifiers used only for splitting
patient_ids = df_analysis["patient_nbr"]

# Get unique patients and their corresponding 30-day readmission status
patient_level = df_analysis[
    ["patient_nbr", "readmission_30d"]
].drop_duplicates("patient_nbr")

print("Unique patients:", patient_level["patient_nbr"].nunique())
print("Patient-level rows:", patient_level.shape[0])

In [ ]:
# Split unique patients into training and test groups
train_patients, test_patients = train_test_split(
    patient_level["patient_nbr"],
    test_size=0.20,
    random_state=42
)

print("Training patients:", len(train_patients))
print("Test patients:", len(test_patients))

In [ ]:
# Assign every encounter to the group of its patient
train_mask = df_analysis["patient_nbr"].isin(train_patients)
test_mask = df_analysis["patient_nbr"].isin(test_patients)

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

print("Training encounters:", len(X_train))
print("Test encounters:", len(X_test))

In [ ]:
# Verify that no patient appears in both datasets
train_patient_set = set(df_analysis.loc[train_mask, "patient_nbr"])
test_patient_set = set(df_analysis.loc[test_mask, "patient_nbr"])

overlap = train_patient_set.intersection(test_patient_set)

print("Patients in both train and test:", len(overlap))

In [ ]:
print("Training readmission rate:",
      round(y_train.mean() * 100, 2), "%")

print("Test readmission rate:",
      round(y_test.mean() * 100, 2), "%")

print("\nTraining target counts:")
print(y_train.value_counts())

print("\nTest target counts:")
print(y_test.value_counts())

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## Predictive Modelling

### Preprocessing Pipeline

A preprocessing pipeline is used to ensure that transformations are fitted using the training data only and then applied to the test data.

Numeric variables are passed through without scaling initially, while categorical variables are one-hot encoded. Unknown categories appearing in the test set are handled safely using `handle_unknown="ignore"`.

The preprocessing is contained within the modelling pipeline to reduce the risk of data leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Identify feature types
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

### Logistic Regression Baseline

Logistic regression is used as the baseline classification model because it provides a relatively simple and interpretable benchmark.

The target is imbalanced, with approximately 11% of encounters followed by a 30-day readmission. Class weighting is therefore used so that the minority readmission class receives greater importance during model fitting.

The model is combined with the preprocessing pipeline so that categorical encoding is fitted only on the training data.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

logistic_model.fit(X_train, y_train)

print("Logistic Regression model fitted successfully.")

### Random Forest

A Random Forest classifier is used as a second model because it can capture nonlinear relationships and interactions between predictors that may not be represented by logistic regression.

Class weighting is used to account for the imbalance between encounters with and without 30-day readmission.

The same preprocessing framework is used so that the comparison between models is based on the same training and test data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

random_forest_model.fit(X_train, y_train)

print("Random Forest model fitted successfully.")

## Model Evaluation

### Test Set Predictions

Both models are evaluated on the held-out test set that was created at patient level.

Class predictions and predicted probabilities are generated for the 30-day readmission outcome. Probabilities are retained because threshold-independent metrics such as ROC-AUC and PR-AUC provide a more informative assessment than accuracy alone for an imbalanced target.

In [ ]:
# Generate class predictions
logistic_predictions = logistic_model.predict(X_test)
random_forest_predictions = random_forest_model.predict(X_test)

# Generate predicted probabilities for the positive class
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]
random_forest_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully.")
print("Test observations:", len(y_test))

### ROC-AUC and PR-AUC

ROC-AUC measures the model's ability to rank positive cases above negative cases across classification thresholds.

Because only 11.16% of encounters resulted in 30-day readmission, precision-recall performance is also examined. PR-AUC is particularly useful for understanding performance on the minority readmission class.

Neither metric depends on selecting a single classification threshold.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

logistic_roc_auc = roc_auc_score(
    y_test,
    logistic_probabilities
)

random_forest_roc_auc = roc_auc_score(
    y_test,
    random_forest_probabilities
)

logistic_pr_auc = average_precision_score(
    y_test,
    logistic_probabilities
)

random_forest_pr_auc = average_precision_score(
    y_test,
    random_forest_probabilities
)

print("Logistic Regression ROC-AUC:",
      round(logistic_roc_auc, 4))

print("Random Forest ROC-AUC:",
      round(random_forest_roc_auc, 4))

print("Logistic Regression PR-AUC:",
      round(logistic_pr_auc, 4))

print("Random Forest PR-AUC:",
      round(random_forest_pr_auc, 4))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

logistic_precision = precision_score(
    y_test,
    logistic_predictions,
    zero_division=0
)

logistic_recall = recall_score(
    y_test,
    logistic_predictions,
    zero_division=0
)

logistic_f1 = f1_score(
    y_test,
    logistic_predictions,
    zero_division=0
)

random_forest_precision = precision_score(
    y_test,
    random_forest_predictions,
    zero_division=0
)

random_forest_recall = recall_score(
    y_test,
    random_forest_predictions,
    zero_division=0
)

random_forest_f1 = f1_score(
    y_test,
    random_forest_predictions,
    zero_division=0
)

print("Logistic Regression")
print("Precision:", round(logistic_precision, 4))
print("Recall:", round(logistic_recall, 4))
print("F1:", round(logistic_f1, 4))

print("\nRandom Forest")
print("Precision:", round(random_forest_precision, 4))
print("Recall:", round(random_forest_recall, 4))
print("F1:", round(random_forest_f1, 4))

### Confusion Matrices

Confusion matrices are used to examine the classification errors made by each model at the default probability threshold of 0.5.

The four outcomes are:

- True positive: correctly identified 30-day readmission
- False positive: predicted readmission but no 30-day readmission occurred
- True negative: correctly identified no 30-day readmission
- False negative: readmission occurred but was not identified by the model

In [ ]:
from sklearn.metrics import confusion_matrix

logistic_cm = confusion_matrix(
    y_test,
    logistic_predictions
)

random_forest_cm = confusion_matrix(
    y_test,
    random_forest_predictions
)

print("Logistic Regression confusion matrix:")
print(logistic_cm)

print("\nRandom Forest confusion matrix:")
print(random_forest_cm)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report
)

logistic_specificity = (
    logistic_cm[0, 0] /
    (logistic_cm[0, 0] + logistic_cm[0, 1])
)

random_forest_specificity = (
    random_forest_cm[0, 0] /
    (random_forest_cm[0, 0] + random_forest_cm[0, 1])
)

logistic_balanced_accuracy = balanced_accuracy_score(
    y_test,
    logistic_predictions
)

random_forest_balanced_accuracy = balanced_accuracy_score(
    y_test,
    random_forest_predictions
)

print("Logistic Regression specificity:", logistic_specificity)
print("Logistic Regression balanced accuracy:", logistic_balanced_accuracy)

print("\nRandom Forest specificity:", random_forest_specificity)
print("Random Forest balanced accuracy:", random_forest_balanced_accuracy)

print("\nLogistic Regression classification report:")
print(classification_report(y_test, logistic_predictions))

print("\nRandom Forest classification report:")
print(classification_report(y_test, random_forest_predictions))

In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "ROC-AUC": [
        0.6439,
        0.6516
    ],
    "PR-AUC": [
        0.1922,
        0.1923
    ],
    "Precision": [
        0.1632,
        0.1738
    ],
    "Recall": [
        0.5426,
        0.4716
    ],
    "F1": [
        0.2510,
        0.2540
    ],
    "Specificity": [
        logistic_specificity,
        random_forest_specificity
    ],
    "Balanced Accuracy": [
        logistic_balanced_accuracy,
        random_forest_balanced_accuracy
    ]
})

model_comparison

In [ ]:
model_comparison.to_csv(
    "../data/cleaned/model_performance_comparison.csv",
    index=False
)

model_comparison

### Model Evaluation Findings

Both models showed modest discrimination between encounters followed and not followed by 30-day readmission. Random Forest achieved a slightly higher ROC-AUC (0.652) than Logistic Regression (0.644), while PR-AUC was almost identical (0.192 for both models).

At the default 0.5 classification threshold, Logistic Regression achieved higher recall (54.3%) than Random Forest (47.2%), identifying more encounters that subsequently resulted in readmission. However, Logistic Regression also produced more false positives and therefore had lower specificity (66.7%) than Random Forest (73.2%).

Random Forest achieved slightly higher precision (17.4%) and F1 score (25.4%) than Logistic Regression (16.3% and 25.1%), although the differences were small. Balanced accuracy was also very similar between the models (60.5% for Logistic Regression and 60.2% for Random Forest).

Overall, neither model demonstrated strong enough predictive performance to support standalone clinical decision-making. Logistic Regression may be preferable where identifying a larger proportion of potential readmissions is prioritised, whereas Random Forest provides better specificity and slightly better overall ranking performance. These results should be interpreted as an educational predictive modelling exercise rather than evidence of clinical utility.

## Model Interpretation

In [ ]:
# Extract Logistic Regression coefficients

logistic_preprocessor = logistic_model.named_steps["preprocessor"]
logistic_classifier = logistic_model.named_steps["classifier"]

feature_names = logistic_preprocessor.get_feature_names_out()
coefficients = logistic_classifier.coef_[0]

logistic_coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "odds_ratio": np.exp(coefficients)
})

logistic_coefficients = logistic_coefficients.sort_values(
    "coefficient",
    ascending=False
)

logistic_coefficients.head(20)

In [ ]:
print("Top positive coefficients:")
display(logistic_coefficients.head(15))

print("\nTop negative coefficients:")
display(logistic_coefficients.tail(15))

In [ ]:
# Extract Random Forest feature importances

rf_preprocessor = random_forest_model.named_steps["preprocessor"]
rf_classifier = random_forest_model.named_steps["classifier"]

rf_feature_names = rf_preprocessor.get_feature_names_out()
rf_importances = rf_classifier.feature_importances_

rf_feature_importance = pd.DataFrame({
    "feature": rf_feature_names,
    "importance": rf_importances
}).sort_values(
    "importance",
    ascending=False
)

display(rf_feature_importance.head(20))

In [ ]:
# Aggregate Random Forest importance back to original features

def get_original_feature(feature_name):
    if feature_name.startswith("num__"):
        return feature_name.replace("num__", "")
    
    if feature_name.startswith("cat__"):
        feature_name = feature_name.replace("cat__", "")
        return feature_name.split("_")[0]
    
    return feature_name


rf_feature_importance["original_feature"] = (
    rf_feature_importance["feature"]
    .apply(get_original_feature)
)

rf_aggregated_importance = (
    rf_feature_importance
    .groupby("original_feature")["importance"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

display(rf_aggregated_importance.head(20))

In [ ]:
# Aggregate Random Forest importance using the original feature names

def map_to_original_feature(feature_name):
    if feature_name.startswith("num__"):
        return feature_name.replace("num__", "")
    
    if feature_name.startswith("cat__"):
        categorical_name = feature_name.replace("cat__", "")
        
        categorical_original_features = [
            "race",
            "gender",
            "age",
            "payer_code",
            "medical_specialty",
            "max_glu_serum",
            "A1Cresult",
            "change",
            "diabetesMed",
            "diag_1_category",
            "diag_2_category",
            "diag_3_category"
        ]
        
        for original_feature in categorical_original_features:
            if categorical_name.startswith(original_feature + "_"):
                return original_feature
        
        # Medication features
        medication_features = [
            "metformin",
            "repaglinide",
            "nateglinide",
            "chlorpropamide",
            "glimepiride",
            "glipizide",
            "glyburide",
            "tolbutamide",
            "pioglitazone",
            "rosiglitazone",
            "acarbose",
            "miglitol",
            "troglitazone",
            "tolazamide",
            "insulin",
            "glyburide-metformin",
            "glipizide-metformin",
            "glimepiride-pioglitazone",
            "metformin-rosiglitazone",
            "metformin-pioglitazone"
        ]
        
        for original_feature in medication_features:
            if categorical_name.startswith(original_feature + "_"):
                return original_feature
    
    return feature_name


rf_feature_importance["original_feature"] = (
    rf_feature_importance["feature"]
    .apply(map_to_original_feature)
)

rf_aggregated_importance = (
    rf_feature_importance
    .groupby("original_feature")["importance"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

display(rf_aggregated_importance.head(20))

In [ ]:
logistic_coefficients.to_csv(
    "../data/cleaned/logistic_regression_coefficients.csv",
    index=False
)

rf_feature_importance.to_csv(
    "../data/cleaned/random_forest_feature_importance.csv",
    index=False
)

rf_aggregated_importance.to_csv(
    "../data/cleaned/random_forest_aggregated_importance.csv",
    index=False
)

### Model Interpretation Findings

The Random Forest model identified prior inpatient utilisation (`number_inpatient`) as the most important predictor of 30-day readmission, accounting for approximately 19.8% of total feature importance. Other relatively important variables included medical specialty, primary and secondary diagnosis categories, length of stay, medication count, laboratory procedures, payer code and prior emergency utilisation.

The importance of prior inpatient utilisation was consistent with the exploratory and statistical analysis performed earlier in the project. It had the largest continuous-variable effect size in the exploratory analysis (Cohen's d = 0.532), suggesting that previous inpatient healthcare utilisation contains useful information for identifying encounters at higher risk of subsequent readmission.

The Logistic Regression coefficients identified several medical specialty, medication and diagnosis categories with relatively large positive or negative associations with predicted readmission odds. However, these coefficients should not be interpreted as causal effects. Category-level estimates may also be influenced by patient mix, category frequency and the choice of reference category.

Overall, the two modelling approaches provide complementary interpretations. Logistic Regression offers interpretable direction and relative odds for individual encoded features, while Random Forest captures nonlinear relationships and identifies the variables contributing most to predictive performance.

## Limitations
Several limitations should be considered when interpreting these results.

1. **Historical dataset:** The dataset contains hospital encounters from 1999–2008. Clinical practice, treatment patterns, coding systems and healthcare delivery have changed substantially since then, limiting the generalisability of the models to current clinical settings.

2. **Encounter-level prediction:** The analysis is based on hospital encounters rather than independent patients. Although a patient-level train/test split was used to reduce leakage between the training and test sets, individual patients can still contribute multiple encounters within the same partition.

3. **Modest predictive performance:** Both models demonstrated only moderate discrimination, with ROC-AUC values of 0.644 for Logistic Regression and 0.652 for Random Forest. These results are not sufficient to support standalone clinical decision-making.

4. **Class imbalance:** Only 11.16% of encounters in the full dataset resulted in a 30-day readmission. Class-weighted models were therefore used, and multiple evaluation metrics were reported rather than relying on accuracy alone.

5. **Missing information:** Several variables contained substantial missingness. Some missing values were treated as an explicit "Unknown" or "Not measured" category, which may itself contain information about clinical workflows rather than simply representing random missing data.

6. **Feature availability:** The modelling design attempted to restrict predictors to information available during the encounter before discharge. Discharge disposition was excluded because it may represent information that would not be available at the intended prediction point.

7. **Model interpretability:** Logistic Regression coefficients and Random Forest feature importance describe associations used by the models and should not be interpreted as causal relationships.

8. **Diagnosis representation:** Detailed ICD-9 diagnosis codes were converted into broader diagnostic categories for modelling. This reduces dimensionality but may also remove clinically relevant detail.

9. **External validation:** The models were evaluated on a held-out patient-level test set from the same historical dataset. Performance on a different hospital system, time period or patient population was not assessed.

10. **Educational rather than clinical use:** This project demonstrates a healthcare analytics and predictive modelling workflow. The models should not be used to make real clinical decisions without substantially more rigorous validation, calibration, fairness assessment and prospective evaluation.

## Conclusion

This project analysed 101,766 hospital encounters from the Diabetes 130-US Hospitals dataset to investigate factors associated with 30-day readmission and develop baseline predictive models.

The exploratory analysis found that prior healthcare utilisation, particularly previous inpatient admissions, showed the strongest association with 30-day readmission. Patients who experienced a 30-day readmission had substantially higher prior inpatient utilisation than other encounters, with a moderate standardised difference (Cohen's d = 0.532).

Two predictive models were developed using a patient-level train/test split: Logistic Regression and Random Forest. Both models demonstrated modest predictive discrimination. Random Forest achieved a slightly higher ROC-AUC (0.652) than Logistic Regression (0.644), while PR-AUC was almost identical between the models (0.192). At the default classification threshold, Logistic Regression achieved higher recall (54.3%), while Random Forest achieved higher specificity (73.2%) and a marginally higher F1 score (0.254 versus 0.251).

Model interpretation reinforced the exploratory findings. Prior inpatient utilisation was the most important Random Forest feature, accounting for approximately 19.8% of total feature importance. Other important predictors included medical specialty, diagnosis categories, length of stay, medication count and prior emergency utilisation.

Overall, the analysis demonstrates a complete healthcare analytics workflow covering data cleaning, exploratory analysis, statistical testing, feature engineering, patient-level model validation and model interpretation. The results suggest that previous healthcare utilisation contains useful information for identifying encounters associated with subsequent readmission, but the predictive performance observed in this historical dataset is not sufficient for standalone clinical decision-making.